# Time dependent analysis with a custom lightcurve

This tutorial shows how to search for neutrinos from a source with a known, fixed time profile, e.g. a model neutrino lightcurve of a supernova, using the public IceTracks-DR2 (14-year) data release.

In contrast to the [untriggered flare search](time_dependent_point_source.ipynb), the time profile is not fitted. Hence, no expectation maximization is needed and the likelihood is maximized directly in the number of signal events, ``ns``, and the spectral index, ``gamma``.

The flux of the source is modeled as

$$\Phi(E,t) = \Phi_0 \left(\frac{E}{E_0}\right)^{-\gamma} T(t),$$

where $T(t)$ is the time flux profile, i.e. the lightcurve. Any time profile can be used, e.g. a tabulated lightcurve via the ``TabulatedTimeFluxProfile`` class.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import skyllh
from skyllh.analyses.i3.publicdata_ps.time_dependent_ps import (
    create_analysis,
    get_signal_time_pdfs,
)
from skyllh.core.config import Config
from skyllh.core.flux_model import TabulatedTimeFluxProfile
from skyllh.core.random import RandomStateService
from skyllh.core.source_model import PointLikeSource

## Defining the lightcurve

The lightcurve is given as tabulated values at given times. The times must be given in MJD. Between the tabulated times the lightcurve is interpolated linearly, outside the tabulated time range it is zero.

As an example we use a fast-rise exponential-decay lightcurve with a rise time of 0.5 days and a decay time of 5 days. The values are relative to the peak value.

In [ ]:
cfg = Config()

t_peak = 58000.0
t = np.concatenate((np.linspace(t_peak - 2.5, t_peak, 50), np.linspace(t_peak, t_peak + 50, 200)[1:]))
values = np.where(t < t_peak, np.exp((t - t_peak) / 0.5), np.exp(-(t - t_peak) / 5.0))
values[0] = values[-1] = 0

lightcurve = TabulatedTimeFluxProfile(t=t, values=values, cfg=cfg)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
tt = np.linspace(lightcurve.t_start - 2, lightcurve.t_stop + 2, 1000)
ax.plot(tt, lightcurve(t=tt))
ax.set_xlabel('MJD')
ax.set_ylabel('T(t)')
plt.show()

Instead of a tabulated lightcurve any other time profile can be used, by deriving a class from ``skyllh.core.flux_model.TimeFluxProfile``. Such a class must implement the profile function, ``__call__``, its integral, ``get_integral``, and the ``move`` method. A ``cdf`` method is optional.

## Creating the analysis

We use all datasets of the IceTracks-DR2 release. The signal is distributed over the datasets according to their effective area and their detector on-time during the lightcurve. Datasets without detector on-time during the lightcurve do not contribute any signal.

**Note:** The datasets of an analysis must not contain the same events. For example, the combined dataset ``IC86_I-XI`` must not be used together with one of its individual seasons, e.g. ``IC86_VII``. Datasets recorded at the same time, e.g. tracks and cascades, must have their common events removed beforehand.

Because the lightcurve is fixed, we can select only the events within the time range of the lightcurve with ``evt_sel_time_window=True``. This speeds up the likelihood evaluation significantly.

In [ ]:
datasets = skyllh.create_datasets('IceTracks-DR2', cfg=cfg)

source = PointLikeSource(name='my source', ra=np.radians(77.36), dec=np.radians(5.69))

ana = create_analysis(
    cfg=cfg,
    datasets=datasets,
    source=source,
    time_flux_profile=lightcurve,
    evt_sel_time_window=True,
)

The fraction of the expected signal events in each dataset can be inspected via the dataset signal weights for a given spectral index:

In [ ]:
src_params_recarray = ana.pmm.create_src_params_recarray(gflp_values=np.array([0, 2.0]))
ana.src_detsigyield_weights_service.calculate(src_params_recarray=src_params_recarray)
ana.ds_sig_weight_factors_service.calculate()
(ds_weights, _) = ana.ds_sig_weight_factors_service.get_weights()
for ds, w in zip(ana.dataset_list, ds_weights, strict=True):
    print(f'{ds.name}: {w:.3f}')

## Unblinding

In [ ]:
(ts, fitparam_values, status) = ana.unblind(minimizer_rss=RandomStateService(seed=1))
print(f'TS = {ts:g}')
print(f'ns = {fitparam_values["ns"]:g}')
print(f'gamma = {fitparam_values["gamma"]:g}')

## Background and signal trials

Background trials are generated by scrambling the event times according to the detector on-time. Signal events are injected with times following the lightcurve during the detector on-time.

In [ ]:
rss = RandomStateService(seed=1)
bkg_trials = ana.do_trials(rss=rss, n=1000, mean_n_sig=0)
sig_trials = ana.do_trials(rss=rss, n=200, mean_n_sig=5)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, max(np.max(sig_trials['ts']), ts) + 1, 40)
ax.hist(bkg_trials['ts'], bins=bins, histtype='step', label='background')
ax.hist(sig_trials['ts'], bins=bins, histtype='step', label='5 signal events')
ax.axvline(ts, color='C2', linestyle='--', label='unblinded')
ax.set_yscale('log')
ax.set_xlabel('TS')
ax.legend()
plt.show()

print(f'mean injected signal events: {np.mean(sig_trials["n_sig"]):.2f}')
print(f'mean fitted signal events: {np.mean(sig_trials["ns"]):.2f}')

## Flux normalization

The mean number of signal events can be converted into the flux normalization $\Phi_0$ of the flux model. Because the flux is $\Phi_0 (E/E_0)^{-\gamma} T(t)$, $\Phi_0$ is the flux at the time when the lightcurve is 1, i.e. the peak flux for the lightcurve defined above. The default units are GeV$^{-1}$ cm$^{-2}$ s$^{-1}$ at $E_0$ = 1 TeV.

The time-integrated flux, i.e. the fluence, is $\Phi_0 \int T(t) dt$, where the lightcurve times are given in days.

In [ ]:
flux_norm = ana.mu2flux(5.0)
fluence = flux_norm * lightcurve.get_total_integral() * 86400
print(f'Phi0 for 5 signal events: {flux_norm:.3e} (GeV cm^2 s)^-1')
print(f'Fluence for 5 signal events: {fluence:.3e} (GeV cm^2)^-1')

## Injecting signal with a different time profile

The time profile of the injected signal can differ from the time profile of the tested signal hypothesis, e.g. to test the robustness of the analysis against a mis-modeled lightcurve. The injection profile is specified via the ``injection_time_flux_profile`` argument of ``create_analysis``, or changed afterwards via the ``change_injection_time_flux_profile`` function. The signal events are then distributed over the datasets according to the injection profile.

The time profile of the signal hypothesis of all datasets can be changed via the ``change_time_flux_profile`` function. This is not possible if the analysis was created with ``evt_sel_time_window=True``.

In [ ]:
for pdf in get_signal_time_pdfs(ana):
    print(pdf.livetime, pdf.time_flux_profile)